# Clustering Evaluation Pipeline

This notebook benchmarks different combinations of **text vectorization**, **dimensionality reduction**, **n_components** (SVD, UMAP parameter), and **clustering algorithms** on arXiv paper datasets.

The goal is to find the configuration that produces the best clustering and coherence metrics for each dataset (8 in total).

**Pipeline overview:**
1. Load and preprocess papers (title + abstract)
2. Vectorize texts (TF-IDF, MiniLM, SciBERT)
3. Reduce dimensionality (SVD, UMAP)
4. Run clustering algorithms (FCM, KMeans, Agglomerative)
5. Evaluate results with CHI, DBI, Silhouette, CV Coherence, and NPMI Coherence 
6. Export best configurations

`seed = 42` is fixed globally for reproducibility.

In [1]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skfuzzy as fuzz

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from umap import UMAP

from transformers import AutoTokenizer, AutoModel, logging as hf_logging
import torch

from metrics_utils import *
from preprocessing_utils import ensure_nltk_resources, preprocess_texts

ensure_nltk_resources()
hf_logging.set_verbosity_error()

seed = 42
np.random.seed(seed)

/home/bernardod/desktop/paper-searcher/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Dataset Registry

Each dataset corresponds to a specific arXiv category and research topic, covering a 3-year window of papers, except Portfolio Optimization and Neural Dynamics (4-year window of papers).

| Category | Topic |
|---|---|
| CS | Trustworthy AI |
| PHYS | Condensed Matter / Superconductivity |
| MATH | Fluid Dynamics |
| EESS | Medical Image Processing |
| STAT | Statistical Machine Learning |
| Q_BIO | Neural Dynamics |
| Q_FIN | Portfolio Optimization |
| ECON | Econometric Modeling |

In [2]:
from enum import Enum

class Categories(Enum):
    CS = "Computer Science"
    PHYS = "Physics"
    MATH = "Mathematics"
    EESS = "Electrical Eng & Systems Sci"
    STAT = "Statistics"
    Q_BIO = "Quantitative Biology"
    Q_FIN = "Quantitative Finance"
    ECON = "Economics"

class DimReduction(Enum):
    UMAP = "UMAP"
    SVD = "SVD"

BASE_PATH = "../datasets/new"

DATASETS = {
    Categories.CS: f"{BASE_PATH}/papers_20230218-20260218_trustworthy_AI.csv",
    Categories.PHYS: f"{BASE_PATH}/papers_20230217-20260217_Condensed_Matter_Superconductivity.csv",
    Categories.MATH: f"{BASE_PATH}/papers_20230218-20260218_Fluid_Dynamics.csv",
    Categories.EESS: f"{BASE_PATH}/papers_20230224-20260224_medical_image_processing.csv",
    Categories.STAT: f"{BASE_PATH}/papers_20230224-20260224_statistical_machine_learning.csv",
    Categories.Q_BIO: f"{BASE_PATH}/papers_20220224-20260224_neural_dynamics.csv",
    Categories.Q_FIN: f"{BASE_PATH}/papers_20220224-20260224_Portfolio_optimization.csv",
    Categories.ECON: f"{BASE_PATH}/papers_20230224-20260224_econometric_modeling.csv"
}

## Experiment Configuration

All tunable parameters for the experiment are centralized in `CONFIG`. Modify this cell to change the scope of the run.

| Parameter | Description |
|---|---|
| `category` | Dataset category to load (see `Categories` enum). |
| `title_column` | Column name containing the document title. |
| `abstract_column` | Column name containing the document abstract. |
| `vectorizers_to_run` | List of vectorization methods or embedding models to evaluate. |
| `tfidf` | TF-IDF hyperparameters, including `min_df`, `max_df`, `max_features`, and `ngram_range`. |
| `dim_reductions` | Dimensionality reduction techniques to evaluate (e.g., `SVD`, `UMAP`). |
| `svd.components` | List of component sizes to evaluate for Truncated SVD. |
| `umap.n_neighbors` | Number of neighbors used in UMAP to balance local vs global structure. |
| `umap.n_components` | List of embedding dimensions to evaluate with UMAP. |
| `umap.metric` | Distance metric used by UMAP (e.g., cosine). |
| `K_values` | Range of cluster counts evaluated during clustering. |
| `algorithms` | Clustering algorithms to run (e.g., `fcm`, `kmeans`, `agglomerative`). |

In [ ]:
# This is not necessary to run the notebook
from huggingface_hub import login

login("")

In [4]:
CONFIG = {
    # Set the current experiment category
    "category": Categories.CS,

    "title_column": "title",
    "abstract_column": "abstract",

    "vectorizers_to_run": ["tfidf", "sentence-transformers/all-MiniLM-L6-v2", "allenai/scibert_scivocab_uncased"],

    "tfidf": dict(min_df=3, max_df=0.7, max_features=3000, ngram_range=(1, 3)),
    
    "dim_reductions": [DimReduction.SVD, DimReduction.UMAP],
    
    "svd": {
        "components": [5, 10, 15]
    },
    
    "umap": {
        "n_neighbors": 15, # Default UMAP value
        "n_components": [5, 10, 15],
        "metric": "cosine"
    },

    "K_values": [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15],
    "algorithms": ["fcm", "kmeans", "agglomerative"],
}

In [5]:
from pprint import pprint

print("Experiment configuration:")
pprint(CONFIG)

Experiment configuration:
{'K_values': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15],
 'abstract_column': 'abstract',
 'algorithms': ['fcm', 'kmeans', 'agglomerative'],
 'category': <Categories.CS: 'Computer Science'>,
 'dim_reductions': [<DimReduction.SVD: 'SVD'>, <DimReduction.UMAP: 'UMAP'>],
 'svd': {'components': [5, 10, 15]},
 'tfidf': {'max_df': 0.7,
           'max_features': 3000,
           'min_df': 3,
           'ngram_range': (1, 3)},
 'title_column': 'title',
 'umap': {'metric': 'cosine', 'n_components': [5, 10, 15], 'n_neighbors': 15},
 'vectorizers_to_run': ['tfidf',
                        'sentence-transformers/all-MiniLM-L6-v2',
                        'allenai/scibert_scivocab_uncased']}


## Data Loading and Preprocessing

The selected dataset is loaded and texts are built by concatenating the **title** and **abstract** columns.

`preprocess_texts` clean and lemmatize text (with POS tagging enabled via `use_pos=True`).

In [6]:
category = CONFIG["category"]
df = pd.read_csv(DATASETS[category], dtype=str)

In [7]:

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def create_ngram_corpus(texts_clean, max_n=3):
    """ 
    Create a tokenized corpus

    This is CRITICAL for OCTIS coherence metrics (`calculate_coherence_cv` and
    `calculate_coherence_npmi`) to work correctly when topics contain bigrams/trigrams.
    The keywords returned by `class_based_tfidf` must exists in corpus created in this function

    Without the token pattern and ENGLISH STOP_WORDS an error occurred when running experiment with scibert
    """
    final_corpus = []
    
    # Extract only 2 tokens or more.
    token_pattern = re.compile(r"(?u)\b\w\w+\b")
    
    for text in texts_clean:
        # Extracts tokens ignoring 1-character garbage and punctuation.
        raw_tokens = token_pattern.findall(text.lower())

        # Apply stopword removal since we have stop_words='english' 
        # in `class_based_tfidf`.
        clean_tokens = [
            word for word in raw_tokens 
            if word not in ENGLISH_STOP_WORDS
        ]
        
        doc_tokens = list(clean_tokens)
        
        # Build bigrams and trigrams
        for n in range(2, max_n + 1):
            for i in range(len(clean_tokens) - n + 1):
                ngram_string = " ".join(clean_tokens[i : i + n])
                doc_tokens.append(ngram_string)

        final_corpus.append(doc_tokens)
        
    return final_corpus

In [8]:
df.head(1)

,arxiv_id,authors,updated,pdf_link,main_category,categories,title,abstract,published,link
0,2504.07170v2,['Jesse C. Cresswell'],2025-11-03 01:42:55+00:00,https://arxiv.org/pdf/2504.07170v2,Machine Learning,"['Machine Learning', 'Artificial Intelligence']",Trustworthy AI Must Account for Interactions,Trustworthy AI encompasses many aspirational a...,2025-04-09 18:00:00+00:00,http://arxiv.org/abs/2504.07170v2


In [ ]:
texts = (df[CONFIG["title_column"]] + " " + df[CONFIG["abstract_column"]]).tolist()
texts_clean = preprocess_texts(texts, use_pos=True)

# Necessary for coherence metrics
corpus = create_ngram_corpus(texts_clean, max_n=3)

## Vectorization

Three vectorization strategies are supported:

- **TF-IDF**
- **Sentence Transformers**
- **SciBERT**

In [10]:
def tfidf_vectorize(texts):
    vectorizer = TfidfVectorizer(
        ngram_range=CONFIG["tfidf"]["ngram_range"],
        min_df=CONFIG["tfidf"]["min_df"],
        max_df=CONFIG["tfidf"]["max_df"],
        max_features=CONFIG["tfidf"]["max_features"],
        stop_words="english",
    )

    return vectorizer.fit_transform(texts)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_cache = {}

def sentence_transformers_vectorize(model_name, texts):
    if model_name in model_cache:
        model = model_cache[model_name]
    else:
        model = SentenceTransformer(model_name)
        model_cache[model_name] = model

    X = model.encode(texts, show_progress_bar=True)
    X = normalize(X, norm="l2")
    return X

def bert_vectorize(texts, model_name, batch_size=32):
    """Encode texts with a BERT-based model using masked mean pooling."""
    if model_name in model_cache:
        tokenizer, model = model_cache[model_name]
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device)
        model.eval()
        model_cache[model_name] = (tokenizer, model)

    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        with torch.inference_mode():
            output = model(**encoded, return_dict=True)
        
        # Masked mean pooling: average token embeddings, ignoring padding tokens
        attention_mask = encoded["attention_mask"]
        token_embeddings = output.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        embeddings = (token_embeddings * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)
        
        all_embeddings.append(embeddings.cpu().numpy())
    
    X = np.vstack(all_embeddings)
    X = normalize(X, norm="l2")
    return X

The `vectorize` routes to the correct function based on the vectorizer name.

In [11]:
def vectorize(vec_name, texts_clean):
    """Route to the appropriate vectorizer based on the vectorizer name."""
    if vec_name == "tfidf":
        return tfidf_vectorize(texts_clean)
    elif "sentence-transformers" in vec_name:
        return sentence_transformers_vectorize(vec_name, texts_clean)
    else:
        return bert_vectorize(texts_clean, vec_name)

## Dimensionality Reduction
- **SVD**
- **UMAP**

`build_X` prepares the final feature matrix used by all clustering algorithms.

In [12]:
def build_X(X_base, n_comp, dim_reduction_method):
    if dim_reduction_method == DimReduction.SVD:
        svd = TruncatedSVD(n_components=n_comp, random_state=seed)
        X = svd.fit_transform(X_base)
        return normalize(X, norm="l2")

    if dim_reduction_method == DimReduction.UMAP:
        umap_cfg = CONFIG["umap"]

        reducer = UMAP(
            n_neighbors=umap_cfg["n_neighbors"],
            n_components=n_comp,
            metric=umap_cfg["metric"],
            random_state=seed
        )

        return reducer.fit_transform(X_base)

    return X_base

## Clustering Algorithms

- **FCM (Fuzzy C-Means)** – soft clustering; each point has a membership degree per cluster. Returns hard labels (argmax of membership matrix `U`).
- **KMeans** – standard hard clustering with k-means++ initialization. Returns labels.
- **Agglomerative** – hierarchical bottom-up clustering.

In [13]:
def cluster_fcm(X, K, m=1.7, error=0.005, maxiter=1000):
    cntr, U, *_rest, fpc = fuzz.cluster.cmeans(
        X.T, c=K, m=m, error=error, maxiter=maxiter, seed=seed,
    )
    U = U.T
    labels = U.argmax(axis=1)
    return labels, {"U": U}

def cluster_kmeans(X, K, max_iter=300):
    model = KMeans(
        n_clusters=K, random_state=seed, init="k-means++", max_iter=max_iter
    )
    labels = model.fit_predict(X)
    return labels

def cluster_agglomerative(X, K):
    model = AgglomerativeClustering(
        n_clusters=K, metric="cosine", linkage="average"
    )
    labels = model.fit_predict(X)
    return labels



## Evaluation Metrics

Cluster Metrics
| Metric | Abbreviation | Optimum | Description |
|---|---|---|---|
| Calinski-Harabász Index | CHI | ↑ higher is better | Ratio of between-cluster to within-cluster dispersion |
| Davies-Bouldin Index | DBI | ↓ lower is better | Average similarity between each cluster and its most similar neighbor |
| Silhouette Score | SIL | ↑ higher is better | Measures how well each point fits its own cluster vs. the nearest cluster |

Coherence Metrics
| Metric | Optimum | Description |
|---|---|---|
| CV Coherence | ↑ higher is better | Measures semantic similarity of words in a topic using a sliding window, normalized PMI, and cosine similarity |
| NPMI Coherence | ↑ higher is better | Normalized Pointwise Mutual Information between pairs of topic words based on their co-occurrence in the corpus |

**FCM collapse detection** (`diagnose_collapse`): FCM can degenerate into a uniform membership matrix where all points belong equally to every cluster. A run is flagged as collapsed when `entropy_norm > 0.85` and is excluded from the results.

In [14]:
def diagnose_collapse(U):
    """Detect FCM degeneracy: returns True if membership entropy is too high (collapsed solution)."""
    eps = 1e-12
    K = U.shape[1]
    entropy = -np.sum(U * np.log(U + eps), axis=1)
    entropy_norm = float(np.mean(entropy) / np.log(K))
    return entropy_norm > 0.85

def evaluate_all(X, labels, top_keywords):
    # Topics are the top keywords of each cluster
    topics = list(top_keywords.values())
    return {
        "CHI": calculate_chi(X, labels),
        "DBI": calculate_dbi(X, labels),
        "SIL": calculate_silhouette(X, labels),
        "CV_COHERENCE": calculate_coherence_cv(topics=topics, corpus=corpus),
        "NPMI_COHERENCE": calculate_coherence_npmi(topics=topics, corpus=corpus),
    }

### Class-Based TF-IDF for Topic Keyword Extraction

This cell implements a **class-based TF-IDF (c-TF-IDF) keyword extraction pipeline** to identify representative terms for each cluster.

The approach works by **aggregating all documents belonging to the same cluster into a single "super-document"**. TF-IDF is then computed across these super-documents, allowing us to identify terms that are **characteristic of each cluster compared to the others**.

In [ ]:
from typing import Dict, List
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

C_TFIDF = ClassTfidfTransformer()

def _remove_substrings(candidates: List[str], top_k: int) -> List[str]:
    """
    Filter a ranked list of n-gram candidates by removing entries that are 
    strictly substrings of another longer candidate, preserving the most 
    specific terms. Returns the top_k survivors.

    Example:
        ["neural network", "network", "deep neural network", "learning"]
        -> ["deep neural network", "learning"]  (with top_k=2)
    """
    final = []
    for word in candidates:
        is_substring = any(
            word != other and word in other for other in candidates
        )
        if not is_substring:
            final.append(word)

        if len(final) >= top_k:
            break

    return final

def class_based_tfidf(texts, labels, top_k=4, ngram_range=(2, 3), max_features=3000, max_df=0.7):
    labels = np.asarray(labels)
    unique_labels = sorted(set(labels.tolist()))

    super_docs = [
        " ".join(texts[i] for i in np.where(labels == lbl)[0])
        for lbl in unique_labels
    ]

    vectorizer = CountVectorizer(
        stop_words="english",
        max_df=max_df,
        ngram_range=ngram_range,
        max_features=max_features,
    )

    X_counts = vectorizer.fit_transform(super_docs)

    X_ctfidf = C_TFIDF.fit_transform(X_counts)
    feature_names = vectorizer.get_feature_names_out()
    top_words = {}

    for i, lbl in enumerate(unique_labels):
        scores = X_ctfidf[i].toarray().ravel()
        ranked_indices = scores.argsort()[::-1]
        candidates = [
            feature_names[j]
            for j in ranked_indices[: top_k * 10]
            if scores[j] > 0
        ]
        top_words[lbl] = _remove_substrings(candidates, top_k)

    return top_words

In [16]:
def make_base_row(vec_name, method, n_comp, K):
    return {
        "vectorizer": vec_name,
        "dim_reduction": method.name,
        "n_comp": n_comp,
        "K": K,
        "collapsed": False,
    }

# Evaluates a single clustering run and appends a result row to the rows list.
def append_result(X, base_row, alg_name, labels, extras, top_keywords):
    extras = extras or {}
    metrics = evaluate_all(X, labels, top_keywords)
    row = {**base_row, "alg": alg_name, **metrics, **extras, "top_keywords": top_keywords}
    rows.append(row)

## Main Experiment Loop

This is the core of the benchmark. For each combination of:

- **Vectorizer** → **Dimensionality Reduction** → **n_components** → **K** → **Algorithm**

The pipeline vectorizes the texts, builds the feature matrix, runs clustering, and records metrics.

Results are collected in `results_all` where each row represents one configuration.

In [17]:
rows = []

for vec_name in CONFIG["vectorizers_to_run"]:
    print(f"---- Running vectorizer: {vec_name} ----")

    X_base = vectorize(vec_name, texts_clean)
    
    for dim_method in CONFIG["dim_reductions"]:
        print(f"---- Using {dim_method.name} in dimensionality reduction ----\n")

        if dim_method == DimReduction.SVD:
            n_components_list = CONFIG["svd"]["components"]

        elif dim_method == DimReduction.UMAP:
            n_components_list = CONFIG["umap"]["n_components"]

        for n_comp in n_components_list:
            print(f"---- Using {n_comp} as n_component ----\n")

            X = build_X(X_base, n_comp, dim_method)

            for K in CONFIG["K_values"]:
        
                base_row = make_base_row(vec_name, dim_method, n_comp, K)

                for alg in CONFIG["algorithms"]:
                    if alg == "fcm":
                        labels, extra = cluster_fcm(X, K)
                        extras = {"collapsed": diagnose_collapse(extra["U"])}
                        alg_name = "FCM"

                    elif alg == "kmeans":
                        labels = cluster_kmeans(X, K)
                        alg_name = "KMeans"

                    elif alg == "agglomerative":
                        labels = cluster_agglomerative(X, K)
                        alg_name = "Agglomerative"

                    top_keywords = class_based_tfidf(texts_clean, labels, top_k=4)
                    append_result(X, base_row, alg_name, labels, extras, top_keywords)
                
            print(f"---- {n_comp} as n_component finish ----\n")

        print(f"---- {dim_method.name} dimensionality reduction finish ----\n")
    
    print(f"---- Vectorizer {vec_name} finish ----")

results_all = pd.DataFrame(rows).reset_index(drop=True)

---- Running vectorizer: tfidf ----
---- Using SVD in dimensionality reduction ----

---- Using 5 as n_component ----

---- 5 as n_component finish ----

---- Using 10 as n_component ----

---- 10 as n_component finish ----

---- Using 15 as n_component ----

---- 15 as n_component finish ----

---- SVD dimensionality reduction finish ----

---- Using UMAP in dimensionality reduction ----

---- Using 5 as n_component ----

---- 5 as n_component finish ----

---- Using 10 as n_component ----

---- 10 as n_component finish ----

---- Using 15 as n_component ----

---- 15 as n_component finish ----

---- UMAP dimensionality reduction finish ----

---- Vectorizer tfidf finish ----
---- Running vectorizer: sentence-transformers/all-MiniLM-L6-v2 ----


Batches: 100%|██████████| 10/10 [00:00<00:00, 16.04it/s]


---- Using SVD in dimensionality reduction ----

---- Using 5 as n_component ----

---- 5 as n_component finish ----

---- Using 10 as n_component ----

---- 10 as n_component finish ----

---- Using 15 as n_component ----

---- 15 as n_component finish ----

---- SVD dimensionality reduction finish ----

---- Using UMAP in dimensionality reduction ----

---- Using 5 as n_component ----

---- 5 as n_component finish ----

---- Using 10 as n_component ----

---- 10 as n_component finish ----

---- Using 15 as n_component ----

---- 15 as n_component finish ----

---- UMAP dimensionality reduction finish ----

---- Vectorizer sentence-transformers/all-MiniLM-L6-v2 finish ----
---- Running vectorizer: allenai/scibert_scivocab_uncased ----


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1486.70it/s, Materializing param=pooler.dense.weight]                               


---- Using SVD in dimensionality reduction ----

---- Using 5 as n_component ----

---- 5 as n_component finish ----

---- Using 10 as n_component ----

---- 10 as n_component finish ----

---- Using 15 as n_component ----

---- 15 as n_component finish ----

---- SVD dimensionality reduction finish ----

---- Using UMAP in dimensionality reduction ----

---- Using 5 as n_component ----

---- 5 as n_component finish ----

---- Using 10 as n_component ----

---- 10 as n_component finish ----

---- Using 15 as n_component ----

---- 15 as n_component finish ----

---- UMAP dimensionality reduction finish ----

---- Vectorizer allenai/scibert_scivocab_uncased finish ----


## Results Analysis

FCM runs flagged as collapsed (degenerate membership) are excluded from further analysis.

In [18]:
from os import makedirs
from datetime import date

today = date.today().strftime("%Y-%m-%d")
DEFAULT_SAVE_PATH = f"../results/{today}/{category.name}"

makedirs(DEFAULT_SAVE_PATH, exist_ok=True)

In [19]:
filtered = results_all[~((results_all["alg"] == "FCM") & (results_all["collapsed"] == True))]
filtered.shape

(528, 12)

### Save experiment results

The resulting CSV contains all valid configurations tested for this dataset
This file serves as the **main record of the experiment for the dataset** and allows
reproducing the ranking and summaries without rerunning the full pipeline.

In [20]:
filtered.to_csv(f"{DEFAULT_SAVE_PATH}/results_{category.name}.csv", index=False)

### Dimensionality reduction comparison

This step evaluates how each dimensionality reduction method performs on average for this dataset.

In [21]:
dimred_summary = (
    filtered
    .groupby("dim_reduction")
    .agg({
        "SIL": "mean",
        "CHI": "mean",
        "DBI": "mean",
        "CV_COHERENCE": "mean",
        "NPMI_COHERENCE": "mean"
    })
    .reset_index()
)
dimred_summary.to_csv(f"{DEFAULT_SAVE_PATH}/dimred_summary_{category.name}.csv", index=False)

### Vectorizer comparison

This allows comparing how different embeddings influence cluster quality and
semantic coherence for this dataset.

In [22]:
vectorizer_summary = (
    filtered
    .groupby("vectorizer")
    .agg({
        "SIL": "mean",
        "CHI": "mean",
        "DBI": "mean",
        "CV_COHERENCE": "mean",
        "NPMI_COHERENCE": "mean"
    })
    .reset_index()
)

vectorizer_summary.to_csv(f"{DEFAULT_SAVE_PATH}/vectorizer_summary_{category.name}.csv", index=False)

In [29]:
vectorizer_summary

,vectorizer,SIL,CHI,DBI,CV_COHERENCE,NPMI_COHERENCE
0,allenai/scibert_scivocab_uncased,0.344985,156.411535,1.353226,0.327201,-0.337066
1,sentence-transformers/all-MiniLM-L6-v2,0.385633,149.579085,1.320565,0.343589,-0.325246
2,tfidf,0.400630,110.242803,1.292991,0.325330,-0.320188


### Clustering algorithm comparison

Compares the clustering algorithms used in the experiment.

This analysis helps identify which clustering method tends to produce
better clusters for this dataset.

In [23]:
algorithm_summary = (
    filtered
    .groupby("alg")
    .agg({
        "SIL": "mean",
        "CHI": "mean",
        "DBI": "mean",
        "CV_COHERENCE": "mean",
        "NPMI_COHERENCE": "mean"
    })
    .reset_index()
)   

algorithm_summary.to_csv(f"{DEFAULT_SAVE_PATH}/algorithm_summary_{category.name}.csv", index=False)

In [28]:
algorithm_summary

,alg,SIL,CHI,DBI,CV_COHERENCE,NPMI_COHERENCE
0,Agglomerative,0.352550,113.037876,1.271763,0.359948,-0.295558
1,FCM,0.419918,184.013404,1.231179,0.316087,-0.345660
2,KMeans,0.373058,134.271785,1.433480,0.314766,-0.347336


### Configuration ranking (rank aggregation)

This step ranks all tested configurations using **rank aggregation across multiple metrics**.

- SIL (higher is better)
- CHI (higher is better)
- DBI (lower is better)
- CV coherence (higher is better)
- NPMI coherence (higher is better)

Configurations are then sorted by `rank_total`, where **lower values indicate
better overall performance**.

The resulting file contains the full ranked list of configurations. We select
the rank 1 (first row) to be used in `pipeline_generalization_analysis.ipynb`.


In [24]:
ranking_df = filtered.copy()

ranking_df["rank_SIL"] = ranking_df["SIL"].rank(ascending=False)
ranking_df["rank_CHI"] = ranking_df["CHI"].rank(ascending=False)
ranking_df["rank_DBI"] = ranking_df["DBI"].rank(ascending=True)
ranking_df["rank_CV"] = ranking_df["CV_COHERENCE"].rank(ascending=False)
ranking_df["rank_NPMI"] = ranking_df["NPMI_COHERENCE"].rank(ascending=False)

ranking_df["rank_total"] = (
    ranking_df["rank_SIL"]
    + ranking_df["rank_CHI"]
    + ranking_df["rank_DBI"]
    + ranking_df["rank_CV"]
    + ranking_df["rank_NPMI"]
)

ranking_df = ranking_df.sort_values("rank_total")

ranking_df.to_csv(f"{DEFAULT_SAVE_PATH}/ranking_configs_{category.name}.csv", index=False)

In [25]:
top_configs = ranking_df.head(5)
top_configs.to_csv(f"{DEFAULT_SAVE_PATH}/top5_configs_{category.name}.csv", index=False)

In [26]:
top_configs

,vectorizer,dim_reduction,n_comp,K,collapsed,alg,CHI,DBI,SIL,CV_COHERENCE,NPMI_COHERENCE,top_keywords,rank_SIL,rank_CHI,rank_DBI,rank_CV,rank_NPMI,rank_total
331,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,3,False,KMeans,310.290985,1.024360,0.585291,0.476117,-0.272126,"{0: ['adversarial attack', 'adversarial exampl...",9.0,18.0,35.0,20.0,72.0,154.0
364,sentence-transformers/all-MiniLM-L6-v2,UMAP,15,3,False,KMeans,301.717194,1.021937,0.594092,0.476117,-0.272126,"{0: ['clinical reasoning', 'cross modal', 'med...",6.0,23.0,33.0,20.0,72.0,154.0
363,sentence-transformers/all-MiniLM-L6-v2,UMAP,15,3,False,FCM,300.622070,1.025028,0.588181,0.476117,-0.272126,"{0: ['adversarial attack', 'adversarial exampl...",7.0,24.0,36.0,20.0,72.0,159.0
330,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,3,False,FCM,309.117126,1.018147,0.578733,0.489891,-0.328857,"{0: ['adversarial attack', 'adversarial exampl...",12.0,19.0,31.0,10.0,249.5,321.5
332,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,3,False,Agglomerative,295.433868,1.025042,0.572081,0.489891,-0.328857,"{0: ['adversarial attack', 'adversarial exampl...",17.0,26.0,37.0,10.0,249.5,339.5


In [27]:
for i in top_configs["top_keywords"]:
    print(i)
    print()

{0: ['adversarial attack', 'adversarial example', 'neural network', 'instruction follow'], 1: ['responsible ai', 'ai development', 'ai governance', 'user trust'], 2: ['clinical reasoning', 'cross modal', 'medical imaging', 'medical image segmentation']}

{0: ['clinical reasoning', 'cross modal', 'medical imaging', 'medical image segmentation'], 1: ['responsible ai', 'ai development', 'ai governance', 'user trust'], 2: ['adversarial attack', 'adversarial example', 'neural network', 'instruction follow']}

{0: ['adversarial attack', 'adversarial example', 'neural network', 'instruction follow'], 1: ['responsible ai', 'ai development', 'ai governance', 'user trust'], 2: ['clinical reasoning', 'cross modal', 'medical imaging', 'medical image segmentation']}

{0: ['adversarial attack', 'adversarial example', 'neural network', 'instruction follow'], 1: ['responsible ai', 'ai development', 'ai governance', 'user trust'], 2: ['clinical reasoning', 'cross modal', 'medical imaging', 'mfc bench']